![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas

El propósito de este proyecto es que puedan poner en práctica, en sus respectivos grupos de trabajo, sus conocimientos sobre técnicas de preprocesamiento, modelos predictivos de NLP, y la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 2: Clasificación de género de películas"

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 8, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/29c44fce98c747f2a1dfdaf29d4c4965).

En este proyecto se usará un conjunto de datos de géneros de películas. Cada observación contiene el título de una película, su año de lanzamiento, la sinopsis o plot de la película (resumen de la trama) y los géneros a los que pertenece (una película puede pertenercer a más de un género). Por ejemplo:
- Título: 'How to Be a Serial Killer'
- Plot: 'A serial killer decides to teach the secrets of his satisfying career to a video store clerk.'
- Generos: 'Comedy', 'Crime', 'Horror'

La idea es que usen estos datos para predecir la probabilidad de que una película pertenezca, dada la sinopsis, a cada uno de los géneros.

![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/moviegenre.png)

### Librerías

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [18]:
# Importación librerías
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Adicionales
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack
from google.colab import files

### Datos para la predicción de género en películas

In [2]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)

In [3]:
# Visualización datos de entrenamiento
dataTraining.head()

,year,title,plot,genres,rating
3107,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
900,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
6724,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
4704,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
2582,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6


In [4]:
# Visualización datos de test
dataTesting.head()

,year,title,plot
1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ...."
4,1978,Midnight Express,"the true story of billy hayes , an american c..."
5,1996,Primal Fear,martin vail left the chicago da ' s office to ...
6,1950,Crisis,husband and wife americans dr . eugene and mr...
7,1959,The Tingler,the coroner and scientist dr . warren chapin ...


## Ensemble TF-IDF + RL (OneVsRest)+ NB

In [21]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [22]:
# Definición de variable de interés (y)
mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [23]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(dataTraining['text'],
                                                              y,
                                                              test_size=0.2,
                                                              random_state=42)



In [24]:
# Definición y entrenamiento

## TF-IDF
vect = TfidfVectorizer(stop_words='english',
                       max_features=15000,
                       ngram_range=(1,2))

X_train_dtm = vect.fit_transform(X_train_text)

X_test_dtm = vect.transform(X_test_text)

## Regresión Logística
lr = OneVsRestClassifier(LogisticRegression(C=2,max_iter=3000))

lr.fit(X_train_dtm, y_train)

pred_lr = lr.predict_proba(X_test_dtm)

## Naive Bayes Multinomial
nb = OneVsRestClassifier(MultinomialNB(alpha=0.1))

nb.fit(X_train_dtm, y_train)

pred_nb = nb.predict_proba(X_test_dtm)

In [25]:
# Predicción del modelo de clasificación

## Ensemble
pred_ensemble = (0.8 * pred_lr + 0.2 * pred_nb)

auc_TFIDF_RL_NB = roc_auc_score(y_test, pred_ensemble, average='macro')

print('ROC AUC Ensemble TF-IDF + RL (OneVsRest)+ NB:', auc_TFIDF_RL_NB)

ROC AUC Ensemble TF-IDF + RL (OneVsRest)+ NB: 0.8890777461549292


In [26]:
# Transformación variables predictoras X del conjunto de test
X_real = vect.transform(dataTesting['text'])

pred_lr_test = lr.predict_proba(X_real)

pred_nb_test = nb.predict_proba(X_real)

pred_final = (0.8 * pred_lr_test + 0.2 * pred_nb_test)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [27]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(pred_final, index=dataTesting.index, columns=cols)

submission.to_csv('Ensemble TFIDF_RL (OneVsRest)_NB.csv', index_label='ID')

files.download("Ensemble TFIDF_RL (OneVsRest)_NB.csv")
print("Ensemble TFIDF_RL (OneVsRest)_NB.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ensemble TFIDF_RL (OneVsRest)_NB.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.097983     0.063335     0.019165     0.026167  0.313276  0.092109   
4  0.117627     0.033690     0.022907     0.149439  0.270935  0.263628   
5  0.081277     0.018597     0.007469     0.051897  0.127465  0.658406   
6  0.062611     0.049712     0.008795     0.037164  0.157816  0.041908   
7  0.032787     0.035722     0.022129     0.016270  0.198986  0.096925   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.023769  0.554706  0.038718   0.111854  ...   0.033732   0.070314   
4       0.038908  0.784446  0.028160   0.023936  ...   0.019508   0.031342   
5       0.020415  0.804303  0.010966   0.017275  ...   0.013715   0.407397   
6       0.031444  0.790828  0.026861   0.031664  ...   0.025804   0.071058   
7       0.017159  0.232497  0.045190   0.101290  ...   0.020575   0.144565   

     p_News  p_Romance  p_S

## Logistic Regression OneVsRest

In [5]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [6]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [7]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(dataTraining['text'],
                                                              y,
                                                              test_size=0.2,
                                                              random_state=42)

In [11]:
# Definición y entrenamiento

## Capturar palabras y frases
word_vectorizer = TfidfVectorizer(stop_words='english',
                                  max_features=15000,
                                  ngram_range=(1,2),
                                  min_df=3,
                                  max_df=0.9,
                                  sublinear_tf=True)

X_train_word = word_vectorizer.fit_transform(X_train_text)

X_test_word = word_vectorizer.transform(X_test_text)

## Capturar fragmentos de palabras
char_vectorizer = TfidfVectorizer(analyzer='char',
                                  ngram_range=(3,5),
                                  max_features=10000,
                                  sublinear_tf=True)

X_train_char = char_vectorizer.fit_transform(X_train_text)

X_test_char = char_vectorizer.transform(X_test_text)

## Extraer train/test numérico
X_train_num = dataTraining.loc[X_train_text.index, ['year']]
X_test_num = dataTraining.loc[X_test_text.index, ['year']]

## Reemplazar nulos
X_train_num = X_train_num.fillna(0)

X_test_num = X_test_num.fillna(0)

## Escalar
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train_num)

X_test_num_scaled = scaler.transform(X_test_num)

## Combinación de features
X_train_final = hstack([X_train_word,
                        X_train_char,
                        X_train_num_scaled])

X_test_final = hstack([X_test_word,
                       X_test_char,
                       X_test_num_scaled])

# Modelo
clf = OneVsRestClassifier(LogisticRegression(C=4,
                                             solver='liblinear',
                                             max_iter=3000,
                                             class_weight='balanced'))

clf.fit(X_train_final, y_train)

OneVsRestClassifier(estimator=LogisticRegression(C=4, class_weight='balanced',
                                                 max_iter=3000,
                                                 solver='liblinear'))

In [12]:
# Predicción del modelo de clasificación
y_pred = clf.predict_proba(X_test_final)

## Evaluación AUC
auc_RL_OVS = roc_auc_score(y_test, y_pred, average='macro')

print("ROC AUC Logistic Regression OneVsRest:", auc_RL_OVS)

ROC AUC Logistic Regression OneVsRest: 0.8989262970553126


In [13]:
# Transformación variables predictoras X del conjunto de test
X_real_text = dataTesting['text']  #Texto
X_real_word = word_vectorizer.transform(X_real_text) #WORD TF-IDF
X_real_char = char_vectorizer.transform(X_real_text) #CHAR TF-IDF

## Variables numéricas
X_real_num = dataTesting[['year']]

X_real_num = X_real_num.fillna(0)

X_real_num_scaled = scaler.transform(X_real_num)

## Combinar test
X_real_final = hstack([X_real_word,
                       X_real_char,
                       X_real_num_scaled])

y_pred_test = clf.predict_proba(X_real_final)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [17]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test,
                          index=dataTesting.index,
                          columns=cols)

submission.to_csv('LRegression_OneVsRest.csv', index_label='ID')

files.download("LRegression_OneVsRest.csv")
print("LR_OneVsRest.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

LR_OneVsRest.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.228442     0.082054     0.035289     0.039729  0.203531  0.065832   
4  0.098580     0.007558     0.015565     0.529085  0.122503  0.484638   
5  0.083001     0.008333     0.002479     0.187268  0.026360  0.960070   
6  0.104641     0.156610     0.002188     0.085229  0.108844  0.035551   
7  0.014978     0.066463     0.043217     0.016770  0.236090  0.020454   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.010717  0.308710  0.032106   0.173383  ...   0.053695   0.048969   
4       0.023067  0.969156  0.004645   0.004393  ...   0.034962   0.014933   
5       0.012815  0.733143  0.004150   0.017770  ...   0.006817   0.866140   
6       0.001940  0.831187  0.015596   0.024325  ...   0.053866   0.073151   
7       0.001954  0.385149  0.049744   0.167612  ...   0.028355   0.036544   

     p_News  p_Romance  p_Sci-Fi   p_Short   p_

## Random Forest

In [ ]:
# Definición de variables predictoras (X)
vect = CountVectorizer(max_features=1000)
X_dtm = vect.fit_transform(dataTraining['plot'])
X_dtm.shape

(7895, 1000)

In [ ]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

In [ ]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train, X_test, y_train_genres, y_test_genres = train_test_split(X_dtm, y_genres, test_size=0.33, random_state=42)

In [ ]:
# Definición y entrenamiento
clf = OneVsRestClassifier(RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=10, random_state=42))
clf.fit(X_train, y_train_genres)

In [ ]:
# Predicción del modelo de clasificación
y_pred_genres = clf.predict_proba(X_test)

# Impresión del desempeño del modelo
roc_auc_score(y_test_genres, y_pred_genres, average='macro')

In [ ]:
# Transformación variables predictoras X del conjunto de test
X_test_dtm = vect.transform(dataTesting['plot'])

cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy', 'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical', 'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

# Predicción del conjunto de test
y_pred_test_genres = clf.predict_proba(X_test_dtm)

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
res = pd.DataFrame(y_pred_test_genres, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_RF.csv', index_label='ID')
files.download("pred_genres_text_RF.csv")
print("pred_genres_text_RF.csv generado y descargado")
print(res.head())